In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from k_means_constrained import KMeansConstrained

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Find the project root
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "geo_dataset"
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "high_resolution_512_finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [5]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


In [6]:
countries = sorted(
    df["country"].unique()
)

country_to_index = {
    country: index
    for index, country in enumerate(countries)
}

index_to_country = {
    index: country
    for country, index in country_to_index.items()
}

df["country_index"] = df["country"].map(
    country_to_index
)

NUMBER_OF_COUNTRIES = len(countries)

print(country_to_index)

{'Belarus': 0, 'Finland': 1, 'France': 2, 'Germany': 3, 'Iceland': 4, 'Italy': 5, 'Norway': 6, 'Poland': 7, 'Spain': 8, 'Sweden': 9, 'Turkey': 10, 'United_Kingdom': 11}


Validation Split

In [7]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


In [8]:
CELLS_PER_COUNTRY = 8

NUMBER_OF_CELLS = (
    NUMBER_OF_COUNTRIES
    * CELLS_PER_COUNTRY
)

train_df = train_df.copy()
val_df = val_df.copy()

train_df["cell_index"] = -1
val_df["cell_index"] = -1

cell_centres = np.zeros(
    (NUMBER_OF_CELLS, 2),
    dtype=np.float32,
)

cell_to_country = np.zeros(
    NUMBER_OF_CELLS,
    dtype=np.int64,
)

country_constrained_models = {}
country_longitude_scales = {}

Creating balanced geographic cells inside each country

In [9]:
for country, country_index in country_to_index.items():

    train_mask = (
        train_df["country"] == country
    )

    val_mask = (
        val_df["country"] == country
    )

    country_train_coordinates = (
        train_df.loc[
            train_mask,
            ["lat", "lng"],
        ]
        .to_numpy(dtype=np.float32)
    )

    country_val_coordinates = (
        val_df.loc[
            val_mask,
            ["lat", "lng"],
        ]
        .to_numpy(dtype=np.float32)
    )

    # Correct longitude distances for latitude
    mean_latitude = np.mean(
        country_train_coordinates[:, 0]
    )

    longitude_scale = np.cos(
        np.radians(mean_latitude)
    )

    country_train_projected = (
        country_train_coordinates.copy()
    )

    country_val_projected = (
        country_val_coordinates.copy()
    )

    country_train_projected[:, 1] *= (
        longitude_scale
    )

    country_val_projected[:, 1] *= (
        longitude_scale
    )

    number_of_country_images = len(
        country_train_coordinates
    )

    minimum_cell_size = (
        number_of_country_images
        // CELLS_PER_COUNTRY
    )

    maximum_cell_size = int(
        np.ceil(
            number_of_country_images
            / CELLS_PER_COUNTRY
        )
    )

    constrained_kmeans = KMeansConstrained(
        n_clusters=CELLS_PER_COUNTRY,
        size_min=minimum_cell_size,
        size_max=maximum_cell_size,
        random_state=42,
        n_init=10,
        max_iter=100,
    )

    local_train_cells = (
        constrained_kmeans.fit_predict(
            country_train_projected
        )
    )

    # Assign each validation image independently
    # to its nearest learned centre.
    projected_centres = (
        constrained_kmeans.cluster_centers_
    )

    validation_distances = (
        (
            country_val_projected[:, None, :]
            - projected_centres[None, :, :]
        )
        ** 2
    ).sum(axis=2)

    local_val_cells = (
        validation_distances.argmin(axis=1)
    )

    first_cell = (
        country_index
        * CELLS_PER_COUNTRY
    )

    last_cell = (
        first_cell
        + CELLS_PER_COUNTRY
    )

    train_df.loc[
        train_mask,
        "cell_index",
    ] = (
        first_cell
        + local_train_cells
    )

    val_df.loc[
        val_mask,
        "cell_index",
    ] = (
        first_cell
        + local_val_cells
    )

    # Calculate centres using the original
    # latitude and longitude coordinates.
    for local_cell in range(
        CELLS_PER_COUNTRY
    ):

        global_cell = (
            first_cell + local_cell
        )

        assigned_coordinates = (
            country_train_coordinates[
                local_train_cells
                == local_cell
            ]
        )

        cell_centres[
            global_cell
        ] = assigned_coordinates.mean(
            axis=0
        )

    cell_to_country[
        first_cell:last_cell
    ] = country_index

    country_constrained_models[
        country
    ] = constrained_kmeans

    country_longitude_scales[
        country
    ] = longitude_scale

In [10]:
train_df["cell_index"] = (
    train_df["cell_index"].astype(int)
)

val_df["cell_index"] = (
    val_df["cell_index"].astype(int)
)

In [11]:
for country in countries:

    country_counts = (
        train_df.loc[
            train_df["country"] == country,
            "cell_index",
        ]
        .value_counts()
        .sort_index()
    )

    print(
        f"{country}: "
        f"min={country_counts.min()}, "
        f"max={country_counts.max()}, "
        f"total={country_counts.sum()}"
    )

    assert (
        country_counts.max()
        - country_counts.min()
        <= 1
    )

Belarus: min=86, max=87, total=691
Finland: min=100, max=100, total=800
France: min=100, max=100, total=800
Germany: min=100, max=100, total=800
Iceland: min=96, max=96, total=768
Italy: min=100, max=100, total=800
Norway: min=100, max=100, total=800
Poland: min=93, max=94, total=747
Spain: min=100, max=100, total=800
Sweden: min=100, max=100, total=800
Turkey: min=100, max=100, total=800
United_Kingdom: min=100, max=100, total=800


In [12]:
print("Countries:", NUMBER_OF_COUNTRIES)
print("Cells:", NUMBER_OF_CELLS)
print("Cell centres:", cell_centres.shape)
print("Cell-country mapping:", cell_to_country.shape)

Countries: 12
Cells: 96
Cell centres: (96, 2)
Cell-country mapping: (96,)


Model Verification

In [13]:
MODEL_NAME = (
    "apple/mobilevitv2-1.0-imagenet1k-256"
)

INPUT_RESOLUTION = 512

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    size={
        "shortest_edge": INPUT_RESOLUTION,
    },
    crop_size={
        "height": INPUT_RESOLUTION,
        "width": INPUT_RESOLUTION,
    },
)

print("Resize:", processor.size)
print("Crop:", processor.crop_size)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUMBER_OF_COUNTRIES + NUMBER_OF_CELLS,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

Resize: SizeDict(height=None, width=None, longest_edge=None, shortest_edge=512, max_height=None, max_width=None)
Crop: SizeDict(height=512, width=512, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)


[transformers] You passed `num_labels=108` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 49179.14it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([108, 512])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([108])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [17]:
HIGH_RESOLUTION_384_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "high_resolution_384"
)

high_resolution_384_model_path = (
    HIGH_RESOLUTION_384_DIR
    / "best_model.pt"
)

In [18]:
weights_384 = torch.load(
    high_resolution_384_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(weights_384)
model = model.to(device)

print("Loaded best 384 model.")

Loaded best 384 model.


In [19]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,444,245


Normalizing grid cell centres

In [20]:
normalized_cell_centres = (
    cell_centres.copy()
)

normalized_cell_centres[:, 0] /= 90
normalized_cell_centres[:, 1] /= 180

cell_centres_tensor = torch.tensor(
    normalized_cell_centres,
    dtype=torch.float32,
    device=device,
)

In [21]:
cell_to_country_tensor = torch.tensor(
    cell_to_country,
    dtype=torch.long,
    device=device,
)

In [22]:
print(cell_centres_tensor.shape)
print(cell_to_country_tensor.shape)

torch.Size([96, 2])
torch.Size([96])


Image Processor and Dataset

In [23]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        country_index = torch.tensor(
            row["country_index"],
            dtype=torch.long,
        )

        cell_index = torch.tensor(
            row["cell_index"],
            dtype=torch.long,
        )

        return (
            pixel_values,
            coordinates,
            country_index,
            cell_index,
        )

In [24]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [25]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [26]:
(
    images,
    coordinates,
    country_labels,
    cell_labels,
) = next(iter(train_loader))

images = images.to(device)
coordinates = coordinates.to(device)
country_labels = country_labels.to(device)
cell_labels = cell_labels.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)
print("Country labels:", country_labels.shape)
print("Cell labels:", cell_labels.shape)

Images: torch.Size([8, 3, 512, 512])
Coordinates: torch.Size([8, 2])
Country labels: torch.Size([8])
Cell labels: torch.Size([8])


In [27]:
images = images.to(device)

with torch.no_grad():

    test_outputs = model(
        pixel_values=images
    ).logits

print("Outputs:", test_outputs.shape)

Outputs: torch.Size([8, 108])


#Loss and Optimizer

In [28]:
coordinate_loss_function = nn.MSELoss()
country_loss_function = nn.CrossEntropyLoss()
cell_loss_function = nn.CrossEntropyLoss()

COUNTRY_LOSS_WEIGHT = 0.01
CELL_LOSS_WEIGHT = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
)

In [29]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

In [30]:
train_assigned_centres = cell_centres[
    train_df["cell_index"].to_numpy()
]

train_oracle_distances = haversine_km(
    train_df["lat"].to_numpy(),
    train_df["lng"].to_numpy(),
    train_assigned_centres[:, 0],
    train_assigned_centres[:, 1],
)

val_assigned_centres = cell_centres[
    val_df["cell_index"].to_numpy()
]

val_oracle_distances = haversine_km(
    val_df["lat"].to_numpy(),
    val_df["lng"].to_numpy(),
    val_assigned_centres[:, 0],
    val_assigned_centres[:, 1],
)

print("Training oracle metrics")
print(
    "Mean:",
    np.mean(train_oracle_distances),
)
print(
    "Median:",
    np.median(train_oracle_distances),
)

print("\nValidation oracle metrics")
print(
    "Mean:",
    np.mean(val_oracle_distances),
)
print(
    "Median:",
    np.median(val_oracle_distances),
)

Training oracle metrics
Mean: 88.02578314730059
Median: 80.44632111370835

Validation oracle metrics
Mean: 86.56898861163306
Median: 80.17229427951997


In [31]:
# ----------------------------------------
# Evaluate inherited 384 model at 512
# ----------------------------------------

model.eval()

initial_512_predictions = []
initial_512_coordinates = []

with torch.inference_mode():

    for (
        images,
        coordinates,
        country_labels,
        cell_labels,
    ) in tqdm(
        val_loader,
        desc="Initial 512 evaluation",
    ):

        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        # Standard temperatures: 1.0 / 1.0
        country_probabilities = torch.softmax(
            country_logits,
            dim=1,
        )

        cell_probabilities = torch.softmax(
            cell_logits,
            dim=1,
        )

        country_weights_for_cells = (
            country_probabilities[
                :, cell_to_country_tensor
            ]
        )

        gated_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        gated_cell_probabilities = (
            gated_cell_probabilities
            / gated_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

        final_coordinates = (
            gated_cell_probabilities
            @ cell_centres_tensor
        )

        initial_512_predictions.append(
            final_coordinates.cpu().numpy()
        )

        initial_512_coordinates.append(
            coordinates.numpy()
        )

initial_512_predictions = np.concatenate(
    initial_512_predictions
)

initial_512_coordinates = np.concatenate(
    initial_512_coordinates
)

predictions_degrees = (
    initial_512_predictions.copy()
)

coordinates_degrees = (
    initial_512_coordinates.copy()
)

predictions_degrees[:, 0] *= 90
predictions_degrees[:, 1] *= 180

coordinates_degrees[:, 0] *= 90
coordinates_degrees[:, 1] *= 180

initial_512_distances = haversine_km(
    coordinates_degrees[:, 0],
    coordinates_degrees[:, 1],
    predictions_degrees[:, 0],
    predictions_degrees[:, 1],
)

initial_512_mean = float(
    np.mean(initial_512_distances)
)

initial_512_median = float(
    np.median(initial_512_distances)
)

initial_512_within_200 = float(
    np.mean(initial_512_distances < 200)
)

initial_512_within_750 = float(
    np.mean(initial_512_distances < 750)
)

print("\nInherited model at 512 resolution")
print(
    f"Mean distance: "
    f"{initial_512_mean:.1f} km"
)
print(
    f"Median distance: "
    f"{initial_512_median:.1f} km"
)
print(
    f"Within 200 km: "
    f"{initial_512_within_200:.2%}"
)
print(
    f"Within 750 km: "
    f"{initial_512_within_750:.2%}"
)

# Use the inherited model as the initial best
best_median = initial_512_median
best_epoch = 0

# Preserve it in the new 512 output directory
torch.save(
    model.state_dict(),
    best_model_path,
)

print(
    "Saved inherited model as the "
    "initial 512 best model."
)

Initial 512 evaluation:   5%|▌         | 16/294 [00:04<01:24,  3.29it/s]


KeyboardInterrupt: 

Full Train + Validation Loop

In [27]:
start_epoch = 0
END_EPOCH = 15

history = []

#best_median = float("inf")
#best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for (
        images,
        coordinates,
        country_labels,
        cell_labels,
    ) in training_bar:

        images = images.to(device)
        coordinates = coordinates.to(device)
        country_labels = country_labels.to(device)
        cell_labels = cell_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=images
        ).logits

        # Split outputs
        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        # Convert logits into probabilities
        country_probabilities = torch.softmax(
            country_logits,
            dim=1,
        )

        cell_probabilities = torch.softmax(
            cell_logits,
            dim=1,
        )

        # Give every cell its country's probability
        country_weights_for_cells = country_probabilities[
            :, cell_to_country_tensor
        ]

        # Gate cells using country probabilities
        gated_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        # Make gated probabilities sum to 1
        gated_cell_probabilities = (
            gated_cell_probabilities
            / gated_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

        # Weighted average of cell centres
        final_coordinates = (
            gated_cell_probabilities
            @ cell_centres_tensor
        )

        coordinate_loss = coordinate_loss_function(
            final_coordinates,
            coordinates,
        )

        country_loss = country_loss_function(
            country_logits,
            country_labels,
        )

        cell_loss = cell_loss_function(
            cell_logits,
            cell_labels,
        )

        loss = (
            coordinate_loss
            + COUNTRY_LOSS_WEIGHT * country_loss
            + CELL_LOSS_WEIGHT * cell_loss
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0

    all_predictions = []
    all_coordinates = []

    correct_country_predictions = 0
    correct_cell_predictions = 0
    number_of_validation_images = 0

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    with torch.no_grad():

        for (
            images,
            coordinates,
            country_labels,
            cell_labels,
        ) in validation_bar:

            images = images.to(device)
            coordinates = coordinates.to(device)
            country_labels = country_labels.to(device)
            cell_labels = cell_labels.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            # Split outputs
            country_logits = outputs[
                :, :NUMBER_OF_COUNTRIES
            ]

            cell_logits = outputs[
                :, NUMBER_OF_COUNTRIES:
            ]

            # Convert logits into probabilities
            country_probabilities = torch.softmax(
                country_logits,
                dim=1,
            )

            cell_probabilities = torch.softmax(
                cell_logits,
                dim=1,
            )

            # Give every cell its country's probability
            country_weights_for_cells = (
                country_probabilities[
                    :, cell_to_country_tensor
                ]
            )

            # Gate cells using country probabilities
            gated_cell_probabilities = (
                cell_probabilities
                * country_weights_for_cells
            )

            # Make gated probabilities sum to 1
            gated_cell_probabilities = (
                gated_cell_probabilities
                / gated_cell_probabilities.sum(
                    dim=1,
                    keepdim=True,
                ).clamp_min(1e-8)
            )

            # Final coordinate prediction
            final_coordinates = (
                gated_cell_probabilities
                @ cell_centres_tensor
            )

            coordinate_loss = coordinate_loss_function(
                final_coordinates,
                coordinates,
            )

            country_loss = country_loss_function(
                country_logits,
                country_labels,
            )

            cell_loss = cell_loss_function(
                cell_logits,
                cell_labels,
            )

            loss = (
                coordinate_loss
                + COUNTRY_LOSS_WEIGHT * country_loss
                + CELL_LOSS_WEIGHT * cell_loss
            )

            total_validation_loss += loss.item()

            all_predictions.append(
                final_coordinates.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

            predicted_countries = country_logits.argmax(
                dim=1
            )

            predicted_cells = cell_logits.argmax(
                dim=1
            )

            correct_country_predictions += (
                predicted_countries == country_labels
            ).sum().item()

            correct_cell_predictions += (
                predicted_cells == cell_labels
            ).sum().item()

            number_of_validation_images += (
                cell_labels.size(0)
            )

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    country_accuracy = (
        correct_country_predictions
        / number_of_validation_images
    )

    cell_accuracy = (
        correct_cell_predictions
        / number_of_validation_images
    )

    # --------------------
    # Geographic metrics
    # --------------------
    all_predictions = np.concatenate(
        all_predictions
    )

    all_coordinates = np.concatenate(
        all_coordinates
    )

    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    # --------------------
    # Save history
    # --------------------
    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
        "country_accuracy": country_accuracy,
        "cell_accuracy": cell_accuracy,
    })

    # --------------------
    # Display results
    # --------------------
    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")
    print(f"Country accuracy: {country_accuracy:.2%}")
    print(f"Cell accuracy: {cell_accuracy:.2%}")

    # --------------------
    # Save best model
    # --------------------
    if median_distance < best_median:

        best_median = median_distance
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:

        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    # --------------------
    # Save resumable checkpoint
    # --------------------
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_median": best_median,
            "best_epoch": best_epoch,
            "history": history,
            "patience": patience,
            "input_resolution": INPUT_RESOLUTION,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
            "model_name": MODEL_NAME,
            "num_labels": (
                NUMBER_OF_COUNTRIES
                + NUMBER_OF_CELLS
            ),
            "parameter_count": total_params,
            "number_of_countries": (
                NUMBER_OF_COUNTRIES
            ),
            "cells_per_country": (
                CELLS_PER_COUNTRY
            ),
            "cell_construction": (
                "capacity_constrained_kmeans"
            ),

            "number_of_cells": NUMBER_OF_CELLS,
            "cell_centres": (
                cell_centres_tensor
                .detach()
                .cpu()
            ),
            "cell_to_country": (
                cell_to_country_tensor
                .detach()
                .cpu()
            ),
            "country_to_index": country_to_index,
            "index_to_country": index_to_country,
            "country_loss_weight": (
                COUNTRY_LOSS_WEIGHT
            ),
            "cell_loss_weight": (
                CELL_LOSS_WEIGHT
            ),
        },
        checkpoint_path,
    )

    # Preserve history after every epoch
    pd.DataFrame(history).to_csv(
        history_path,
        index=False,
    )

    # --------------------
    # Early stopping
    # --------------------
    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/15 - Validation: 100%|██████████| 294/294 [00:20<00:00, 14.51it/s]



Epoch 1 results
Training loss: 0.0189
Validation loss: 0.0481
Mean distance: 511.6 km
Median distance: 222.7 km
Within 200 km: 46.98%
Within 750 km: 77.47%
Country accuracy: 71.68%
Cell accuracy: 37.93%
Saved new best model.


Epoch 2/15 - Validation: 100%|██████████| 294/294 [00:21<00:00, 13.98it/s]



Epoch 2 results
Training loss: 0.0137
Validation loss: 0.0470
Mean distance: 500.6 km
Median distance: 224.3 km
Within 200 km: 47.24%
Within 750 km: 78.44%
Country accuracy: 72.24%
Cell accuracy: 38.52%
Epochs without improvement: 1


Epoch 3/15 - Validation: 100%|██████████| 294/294 [00:20<00:00, 14.35it/s]


...


Epoch 7 results
Training loss: 0.0072
Validation loss: 0.0492
Mean distance: 513.7 km
Median distance: 217.7 km
Within 200 km: 47.92%
Within 750 km: 78.02%
Country accuracy: 72.07%
Cell accuracy: 39.20%
Epochs without improvement: 2


Epoch 8/15 - Validation: 100%|██████████| 294/294 [00:20<00:00, 14.61it/s]



Epoch 8 results
Training loss: 0.0065
Validation loss: 0.0495
Mean distance: 501.7 km
Median distance: 216.7 km
Within 200 km: 47.75%
Within 750 km: 78.70%
Country accuracy: 73.04%
Cell accuracy: 38.44%
Epochs without improvement: 3


Epoch 9/15 - Validation: 100%|██████████| 294/294 [00:20<00:00, 14.32it/s]



Epoch 9 results
Training loss: 0.0059
Validation loss: 0.0505
Mean distance: 505.2 km
Median distance: 218.0 km
Within 200 km: 47.75%
Within 750 km: 78.53%
Country accuracy: 73.30%
Cell accuracy: 38.69%
Epochs without improvement: 4
Early stopping.


In [28]:
best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

print("Loaded epoch-5 best model.")
print("Resolution:", INPUT_RESOLUTION)

Loaded epoch-5 best model.
Resolution: 512


In [32]:
all_country_logits = []
all_cell_logits = []
all_coordinates = []
all_country_labels = []

model.eval()

with torch.inference_mode():

    for (
        images,
        coordinates,
        country_labels,
        _,
    ) in tqdm(
        val_loader,
        desc="Collecting validation logits",
    ):

        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        all_country_logits.append(
            country_logits.cpu()
        )

        all_cell_logits.append(
            cell_logits.cpu()
        )

        all_coordinates.append(
            coordinates
        )

        all_country_labels.append(
            country_labels
        )

In [33]:
all_country_logits = torch.cat(
    all_country_logits,
    dim=0,
)

all_cell_logits = torch.cat(
    all_cell_logits,
    dim=0,
)

all_coordinates = torch.cat(
    all_coordinates,
    dim=0,
)

all_country_labels = torch.cat(
    all_country_labels,
    dim=0,
)

In [34]:
evaluation_cell_to_country = (
    cell_to_country_tensor
    .detach()
    .cpu()
)

evaluation_cell_centres = (
    cell_centres_tensor
    .detach()
    .cpu()
)

print(
    all_country_logits.device,
    evaluation_cell_to_country.device,
    evaluation_cell_centres.device,
)

cpu cpu cpu


In [35]:
def evaluate_configuration(
    name,
    country_temperature=1.0,
    cell_temperature=1.0,
    use_country_gating=True,
):

    country_probabilities = torch.softmax(
        all_country_logits
        / country_temperature,
        dim=1,
    )

    cell_probabilities = torch.softmax(
        all_cell_logits
        / cell_temperature,
        dim=1,
    )

    if use_country_gating:

        country_weights_for_cells = (
            country_probabilities[
                :, evaluation_cell_to_country
            ]
        )

        final_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        final_cell_probabilities = (
            final_cell_probabilities
            / final_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

    else:

        final_cell_probabilities = (
            cell_probabilities
        )

    predicted_coordinates = (
        final_cell_probabilities
        @ evaluation_cell_centres
    )

    predictions_degrees = (
        predicted_coordinates.numpy().copy()
    )

    coordinates_degrees = (
        all_coordinates.numpy().copy()
    )

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    return {
        "configuration": name,
        "country_temperature": (
            country_temperature
        ),
        "cell_temperature": (
            cell_temperature
        ),
        "country_gating": (
            use_country_gating
        ),
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(
            distances < 200
        ),
        "within_750": np.mean(
            distances < 750
        ),
    }

In [36]:
predicted_countries = (
    all_country_logits.argmax(dim=1)
)

country_accuracy = (
    predicted_countries
    == all_country_labels
).float().mean().item()

print(
    f"Country accuracy: "
    f"{country_accuracy:.2%}"
)

Country accuracy: 67.69%


In [37]:
temperature_values = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
]

temperature_results = []

In [38]:
for country_temperature in temperature_values:

    for cell_temperature in temperature_values:

        result = evaluate_configuration(
            name="Country gating",
            country_temperature=(
                country_temperature
            ),
            cell_temperature=(
                cell_temperature
            ),
            use_country_gating=True,
        )

        temperature_results.append(result)

In [39]:
for cell_temperature in temperature_values:

    result = evaluate_configuration(
        name="No country gating",
        country_temperature=1.0,
        cell_temperature=(
            cell_temperature
        ),
        use_country_gating=False,
    )

    temperature_results.append(result)

In [41]:
temperature_results_df = pd.DataFrame(
    temperature_results
)

temperature_results_df = (
    temperature_results_df
    .sort_values(
        by=[
            "median_km",
            "mean_km",
        ]
    )
    .reset_index(drop=True)
)

display(
    temperature_results_df.head(15)
)

temperature_results_df.to_csv(
    #results_path,
    index=False,
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,1.00,True,584.045532,271.460571,0.412415,0.732568
1,Country gating,0.25,1.25,True,583.435974,273.121857,0.406037,0.732143
2,Country gating,0.50,1.00,True,578.435242,276.134216,0.410714,0.733418
3,Country gating,0.25,0.75,True,585.467346,277.305359,0.415391,0.730867
4,Country gating,0.25,0.50,True,588.051636,277.609680,0.412415,0.727891
5,Country gating,0.50,1.25,True,577.418701,277.886841,0.403486,0.736395
6,Country gating,0.25,2.00,True,584.400452,279.115173,0.405612,0.733418
7,Country gating,0.25,1.50,True,583.274658,279.977783,0.408588,0.732143
8,Country gating,0.75,1.00,True,575.172913,280.134644,0.407313,0.735544
9,Country gating,0.50,0.75,True,580.787598,280.707458,0.411139,0.731718


'configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750\nCountry gating,0.25,1.0,True,584.04553,271.46057,0.41241496598639454,0.7325680272108843\nCountry gating,0.25,1.25,True,583.436,273.12186,0.4060374149659864,0.7321428571428571\nCountry gating,0.5,1.0,True,578.43524,276.13422,0.4107142857142857,0.7334183673469388\nCountry gating,0.25,0.75,True,585.46735,277.30536,0.415391156462585,0.7308673469387755\nCountry gating,0.25,0.5,True,588.05164,277.60968,0.41241496598639454,0.7278911564625851\nCountry gating,0.5,1.25,True,577.4187,277.88684,0.40348639455782315,0.7363945578231292\nCountry gating,0.25,2.0,True,584.40045,279.11517,0.40561224489795916,0.7334183673469388\nCountry gating,0.25,1.5,True,583.27466,279.97778,0.4085884353741497,0.7321428571428571\nCountry gating,0.75,1.0,True,575.1729,280.13464,0.407312925170068,0.7355442176870748\nCountry gating,0.5,0.75,True,580.7876,280.70746,0.4111394557823129,0.73171768707483\nCountry gating,0

In [42]:
best_gated_result = (
    temperature_results_df[
        temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

best_ungated_result = (
    temperature_results_df[
        ~temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

display(
    pd.DataFrame([
        best_gated_result,
        best_ungated_result,
    ])
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,1.00,True,584.045532,271.460571,0.412415,0.732568
45,No country gating,1.00,0.25,False,598.629333,302.402588,0.406463,0.711735


In [43]:
country_temperature_values = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

cell_temperature_values = [
    0.60,
    0.75,
    0.90,
]

In [44]:
local_temperature_results = []

for country_temperature in (
    country_temperature_values
):

    for cell_temperature in (
        cell_temperature_values
    ):

        result = evaluate_configuration(
            name="Country gating",
            country_temperature=(
                country_temperature
            ),
            cell_temperature=(
                cell_temperature
            ),
            use_country_gating=True,
        )

        local_temperature_results.append(
            result
        )

local_temperature_results_df = (
    pd.DataFrame(
        local_temperature_results
    )
    .sort_values(
        by=[
            "median_km",
            "mean_km",
        ]
    )
    .reset_index(drop=True)
)

display(
    local_temperature_results_df
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.20,0.90,True,586.126587,272.510529,0.414541,0.731293
1,Country gating,0.10,0.90,True,589.985229,272.980835,0.416667,0.727891
2,Country gating,0.15,0.90,True,588.035583,273.793030,0.416241,0.730017
3,Country gating,0.15,0.60,True,590.006287,274.526917,0.416667,0.727891
4,Country gating,0.15,0.75,True,588.918762,274.558411,0.416667,0.727891
5,Country gating,0.10,0.75,True,591.303345,274.558411,0.417092,0.727041
6,Country gating,0.25,0.90,True,584.493103,275.412903,0.414116,0.732568
7,Country gating,0.20,0.75,True,587.009583,276.061676,0.416667,0.729592
8,Country gating,0.30,0.60,True,585.831970,276.114746,0.413690,0.728741
9,Country gating,0.10,0.60,True,592.589172,276.758545,0.416241,0.727041


In [45]:
local_temperature_results_df.to_csv(
    OUTPUT_DIR
    / "local_temperature_results.csv",
    index=False,
)

In [46]:
BEST_COUNTRY_TEMPERATURE = 0.10
BEST_CELL_TEMPERATURE = 0.75

Post Processing - multi-resolution inference

In [47]:
country_logits_512 = all_country_logits.clone()
cell_logits_512 = all_cell_logits.clone()
coordinates_512 = all_coordinates.clone()
country_labels_512 = all_country_labels.clone()

print(country_logits_512.shape)
print(cell_logits_512.shape)
print(coordinates_512.shape)

torch.Size([2352, 12])
torch.Size([2352, 96])
torch.Size([2352, 2])


In [48]:
processor_384 = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    size={
        "shortest_edge": 384,
    },
    crop_size={
        "height": 384,
        "width": 384,
    },
)

val_dataset_384 = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor_384,
    transform=None,
)

val_loader_384 = DataLoader(
    val_dataset_384,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [46]:
best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

MobileViTV2ForImageClassification(
  (mobilevitv2): MobileViTV2Model(
    (conv_stem): MobileViTV2ConvLayer(
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (normalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): MobileViTV2Encoder(
      (layer): ModuleList(
        (0): MobileViTV2MobileNetLayer(
          (layer): ModuleList(
            (0): MobileViTV2InvertedResidual(
              (expand_1x1): MobileViTV2ConvLayer(
                (convolution): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
                (activation): SiLU()
              )
              (conv_3x3): MobileViTV2ConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), gr

In [49]:
country_logits_384_batches = []
cell_logits_384_batches = []
coordinates_384_batches = []
country_labels_384_batches = []

with torch.inference_mode():

    for (
        images,
        coordinates,
        country_labels,
        cell_labels,
    ) in tqdm(
        val_loader_384,
        desc="Evaluating at 384",
    ):

        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        country_logits_384_batches.append(
            country_logits.cpu()
        )

        cell_logits_384_batches.append(
            cell_logits.cpu()
        )

        coordinates_384_batches.append(
            coordinates.cpu()
        )

        country_labels_384_batches.append(
            country_labels.cpu()
        )

country_logits_384 = torch.cat(
    country_logits_384_batches
)

cell_logits_384 = torch.cat(
    cell_logits_384_batches
)

coordinates_384 = torch.cat(
    coordinates_384_batches
)

country_labels_384 = torch.cat(
    country_labels_384_batches
)

Evaluating at 384: 100%|██████████| 147/147 [00:08<00:00, 17.39it/s]


In [50]:
assert torch.equal(
    country_labels_384,
    country_labels_512,
)

assert torch.allclose(
    coordinates_384,
    coordinates_512,
)

print("384 and 512 validation ordering matches.")

384 and 512 validation ordering matches.


In [51]:
def evaluate_logits(
    name,
    country_logits,
    cell_logits,
    country_temperature=0.25,
    cell_temperature=0.75,
):

    country_probabilities = torch.softmax(
        country_logits / country_temperature,
        dim=1,
    )

    cell_probabilities = torch.softmax(
        cell_logits / cell_temperature,
        dim=1,
    )

    country_weights_for_cells = (
        country_probabilities[
            :, evaluation_cell_to_country
        ]
    )

    gated_cell_probabilities = (
        cell_probabilities
        * country_weights_for_cells
    )

    gated_cell_probabilities = (
        gated_cell_probabilities
        / gated_cell_probabilities.sum(
            dim=1,
            keepdim=True,
        ).clamp_min(1e-8)
    )

    predictions = (
        gated_cell_probabilities
        @ evaluation_cell_centres
    )

    predictions_degrees = predictions.numpy().copy()
    coordinates_degrees = coordinates_512.numpy().copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    return {
        "configuration": name,
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(distances < 200),
        "within_750": np.mean(distances < 750),
    }

In [52]:
combined_country_logits = (
    0.5 * country_logits_512
    + 0.5 * country_logits_384
)

combined_cell_logits = (
    0.5 * cell_logits_512
    + 0.5 * cell_logits_384
)

multi_resolution_results = [
    evaluate_logits(
        "512 only",
        country_logits_512,
        cell_logits_512,
    ),
    evaluate_logits(
        "384 only",
        country_logits_384,
        cell_logits_384,
    ),
    evaluate_logits(
        "384 + 512 equal average",
        combined_country_logits,
        combined_cell_logits,
    ),
]

display(
    pd.DataFrame(
        multi_resolution_results
    ).sort_values("median_km")
)

,configuration,mean_km,median_km,within_200,within_750
1,384 only,543.818420,226.834167,0.469388,0.756803
2,384 + 512 equal average,540.879028,233.505905,0.456633,0.763180
0,512 only,585.467346,277.305359,0.415391,0.730867


In [53]:
mixing_results = []

for weight_512 in [
    0.25,
    0.50,
    0.75,
]:

    mixed_country_logits = (
        weight_512 * country_logits_512
        + (1 - weight_512) * country_logits_384
    )

    mixed_cell_logits = (
        weight_512 * cell_logits_512
        + (1 - weight_512) * cell_logits_384
    )

    result = evaluate_logits(
        name=f"512 weight {weight_512:.2f}",
        country_logits=mixed_country_logits,
        cell_logits=mixed_cell_logits,
        country_temperature=0.25,
        cell_temperature=0.75,
    )

    result["weight_512"] = weight_512
    mixing_results.append(result)

display(
    pd.DataFrame(mixing_results)
    .sort_values("median_km")
)

,configuration,mean_km,median_km,within_200,within_750,weight_512
0,512 weight 0.25,536.180603,222.511261,0.474065,0.76148,0.25
1,512 weight 0.50,540.879028,233.505905,0.456633,0.76318,0.50
2,512 weight 0.75,554.253418,255.247726,0.442602,0.75085,0.75


In [52]:
final_inference_configuration = {
    "resolutions": [384, 512],
    "resolution_weights": {
        384: 0.50,
        512: 0.50,
    },
    "country_temperature": 0.25,
    "cell_temperature": 0.75,
    "country_gating": True,
    "validation_mean_km": 497.725220,
    "validation_median_km": 199.057251,
    "validation_within_200": 0.500425,
    "validation_within_750": 0.786139,
}

final_inference_configuration

{'resolutions': [384, 512],
 'resolution_weights': {384: 0.5, 512: 0.5},
 'country_temperature': 0.25,
 'cell_temperature': 0.75,
 'country_gating': True,
 'validation_mean_km': 497.72522,
 'validation_median_km': 199.057251,
 'validation_within_200': 0.500425,
 'validation_within_750': 0.786139}

In [54]:
from pathlib import Path
import torch

best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

MobileViTV2ForImageClassification(
  (mobilevitv2): MobileViTV2Model(
    (conv_stem): MobileViTV2ConvLayer(
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (normalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): MobileViTV2Encoder(
      (layer): ModuleList(
        (0): MobileViTV2MobileNetLayer(
          (layer): ModuleList(
            (0): MobileViTV2InvertedResidual(
              (expand_1x1): MobileViTV2ConvLayer(
                (convolution): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
                (activation): SiLU()
              )
              (conv_3x3): MobileViTV2ConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), gr

In [15]:
SUBMISSION_DIR = Path("submission")
FINAL_MODEL_DIR = SUBMISSION_DIR / "model"

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(
    FINAL_MODEL_DIR,
    safe_serialization=True,
)

processor.save_pretrained(
    FINAL_MODEL_DIR,
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]


['submission/model/preprocessor_config.json']

In [55]:
inference_config = {
    "resolutions": [384, 512],
    "resolution_weights": {
        384: 0.50,
        512: 0.50,
    },
    "country_temperature": 0.25,
    "cell_temperature": 0.75,
    "number_of_countries": NUMBER_OF_COUNTRIES,
    "number_of_cells": NUMBER_OF_CELLS,

    # These centres are normalized:
    # latitude divided by 90
    # longitude divided by 180
    "cell_centres": (
        evaluation_cell_centres
        .detach()
        .cpu()
        .float()
    ),

    "cell_to_country": (
        evaluation_cell_to_country
        .detach()
        .cpu()
        .long()
    ),

    "country_to_index": country_to_index,
    "parameter_count": total_params,

    "validation_metrics": {
        "mean_km": 497.725220,
        "median_km": 199.057251,
        "within_200": 0.500425,
        "within_750": 0.786139,
    },
}

torch.save(
    inference_config,
    FINAL_MODEL_DIR / "inference_config.pt",
)

In [56]:
print("Final model directory:")

for path in sorted(FINAL_MODEL_DIR.iterdir()):
    print(
        path.name,
        f"{path.stat().st_size / 1_000_000:.2f} MB",
    )

Final model directory:
config.json 0.01 MB
inference_config.pt 0.00 MB
model.safetensors 17.88 MB
preprocessor_config.json 0.00 MB


In [57]:
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
)

FINAL_MODEL_DIR = Path(
    "submission/model"
)

final_config = torch.load(
    FINAL_MODEL_DIR / "inference_config.pt",
    map_location="cpu",
    weights_only=True,
)

verification_model = (
    AutoModelForImageClassification
    .from_pretrained(FINAL_MODEL_DIR)
    .to(device)
)

verification_model.eval()

print(
    "Parameters:",
    sum(
        parameter.numel()
        for parameter in verification_model.parameters()
    ),
)

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 355.77it/s]


Parameters: 4444245


In [58]:
processor_384 = AutoImageProcessor.from_pretrained(
    FINAL_MODEL_DIR,
    size={
        "shortest_edge": 384,
    },
    crop_size={
        "height": 384,
        "width": 384,
    },
)

processor_512 = AutoImageProcessor.from_pretrained(
    FINAL_MODEL_DIR,
    size={
        "shortest_edge": 512,
    },
    crop_size={
        "height": 512,
        "width": 512,
    },
)

In [59]:
val_dataset_384 = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor_384,
    transform=None,
)

val_dataset_512 = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor_512,
    transform=None,
)

val_loader_384 = DataLoader(
    val_dataset_384,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

val_loader_512 = DataLoader(
    val_dataset_512,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [60]:
def collect_validation_logits(
    model,
    loader,
    description,
):

    country_batches = []
    cell_batches = []
    coordinate_batches = []
    country_label_batches = []

    model.eval()

    with torch.inference_mode():

        for (
            images,
            coordinates,
            country_labels,
            cell_labels,
        ) in tqdm(
            loader,
            desc=description,
        ):

            images = images.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            country_logits = outputs[
                :, :NUMBER_OF_COUNTRIES
            ]

            cell_logits = outputs[
                :, NUMBER_OF_COUNTRIES:
            ]

            country_batches.append(
                country_logits.cpu()
            )

            cell_batches.append(
                cell_logits.cpu()
            )

            coordinate_batches.append(
                coordinates.cpu()
            )

            country_label_batches.append(
                country_labels.cpu()
            )

    return (
        torch.cat(country_batches),
        torch.cat(cell_batches),
        torch.cat(coordinate_batches),
        torch.cat(country_label_batches),
    )

In [61]:
(
    country_logits_384,
    cell_logits_384,
    coordinates_384,
    country_labels_384,
) = collect_validation_logits(
    verification_model,
    val_loader_384,
    "Verifying at 384",
)

(
    country_logits_512,
    cell_logits_512,
    coordinates_512,
    country_labels_512,
) = collect_validation_logits(
    verification_model,
    val_loader_512,
    "Verifying at 512",
)

Verifying at 512: 100%|██████████| 294/294 [00:12<00:00, 22.66it/s]


In [62]:
assert torch.equal(
    country_labels_384,
    country_labels_512,
)

assert torch.allclose(
    coordinates_384,
    coordinates_512,
)

print("Validation ordering matches.")

Validation ordering matches.


In [63]:
evaluation_cell_centres = (
    final_config["cell_centres"]
)

evaluation_cell_to_country = (
    final_config["cell_to_country"]
)

print(
    evaluation_cell_centres.shape,
    evaluation_cell_to_country.shape,
)

torch.Size([96, 2]) torch.Size([96])


In [64]:
combined_country_logits = (
    0.50 * country_logits_384
    + 0.50 * country_logits_512
)

combined_cell_logits = (
    0.50 * cell_logits_384
    + 0.50 * cell_logits_512
)

verification_result = evaluate_logits(
    name="Exported final model",
    country_logits=combined_country_logits,
    cell_logits=combined_cell_logits,
    country_temperature=(
        final_config["country_temperature"]
    ),
    cell_temperature=(
        final_config["cell_temperature"]
    ),
)

display(
    pd.DataFrame([
        verification_result
    ])
)

,configuration,mean_km,median_km,within_200,within_750
0,Exported final model,497.72522,199.057251,0.500425,0.786139


In [65]:
validation_country_probabilities = torch.softmax(
    combined_country_logits / 0.25,
    dim=1,
)

validation_cell_probabilities = torch.softmax(
    combined_cell_logits / 0.75,
    dim=1,
)

validation_country_weights = (
    validation_country_probabilities[
        :, evaluation_cell_to_country
    ]
)

validation_gated_probabilities = (
    validation_cell_probabilities
    * validation_country_weights
)

validation_gated_probabilities = (
    validation_gated_probabilities
    / validation_gated_probabilities.sum(
        dim=1,
        keepdim=True,
    )
)

validation_coordinates = (
    validation_gated_probabilities
    @ evaluation_cell_centres
)

validation_coordinates = (
    validation_coordinates.numpy()
)

validation_coordinates[:, 0] *= 90
validation_coordinates[:, 1] *= 180

In [66]:
VALIDATION_CHECK_DIR = Path(
    "outputs/final_validation_check"
)

VALIDATION_CHECK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [67]:
validation_predictions = pd.DataFrame({
    "filename": val_df["filename"].to_numpy(),
    "pred_lat": validation_coordinates[:, 0],
    "pred_lng": validation_coordinates[:, 1],
})

validation_predictions.to_csv(
    VALIDATION_CHECK_DIR
    / "validation_predictions.csv",
    index=False,
)

In [68]:
validation_labels = val_df[
    ["filename", "lat", "lng"]
].copy()

validation_labels.to_csv(
    VALIDATION_CHECK_DIR
    / "validation_labels.csv",
    index=False,
)

print("Validation files saved.")
print("Predictions:", len(validation_predictions))
print("Labels:", len(validation_labels))

Validation files saved.
Predictions: 2352
Labels: 2352
